# Track B Architecture Comparison

这个 notebook 用来做 Track B 的统一实验台。现在已经接入 `encoder_only`、`cls-token transformer`、`1-D Conv + Transformer`、`TCN + Transformer`、`MAE-style encoder-decoder transformer`、`pure_rnn`、`rnn_lstm_hybrid`、`RNN Autoencoder`、`LSTM Autoencoder`、`Transformer Autoencoder`、`VAE`、`KMeans`、`PCA + KMeans`。

后面如果你还要继续加别的架构，只要沿用同样的 runner 接口注册进去，这个 notebook 就可以继续作为统一对比面板。


In [1]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

LOCAL_MODULE_NAMES = [
    "encoder_only_transformer",
    "rnn_baselines",
    "experiment_utils",
    "advanced_transformer_variants",
    "autoencoder_and_clustering_baselines",
    "experiment_presets",
]

for module_name in LOCAL_MODULE_NAMES:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
    else:
        importlib.import_module(module_name)

from encoder_only_transformer import HMMLEARN_AVAILABLE, compare_experiment_summaries
from experiment_presets import (
    build_architecture_runners,
    build_default_clustering_config,
    build_default_data_config,
    build_default_experiment_setups,
    build_default_hmm_config,
    build_default_model_config,
    build_default_training_config,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)


## Single Experiment Configuration

先在这里配置一个单独实验。当前支持的架构已经不止 RNN / Transformer 基线，而是统一通过 `ARCHITECTURE_RUNNERS` 注册。你只需要切换 `architecture_name`，下面的数据构造、训练、聚类和评估流程都会复用。


In [2]:
ARCHITECTURE_RUNNERS = build_architecture_runners()
TARGET_CLUSTER_COUNT = 4
DATA_DIR = NOTEBOOK_DIR.parent / "data"

architecture_name = "encoder_only"
experiment_name = f"{architecture_name}_baseline"

data_config = build_default_data_config(DATA_DIR)
model_config = build_default_model_config(architecture_name)
training_config = build_default_training_config(architecture_name)
clustering_config = build_default_clustering_config(target_cluster_count=TARGET_CLUSTER_COUNT)
hmm_config = build_default_hmm_config(
    hmm_enabled=HMMLEARN_AVAILABLE,
    target_cluster_count=TARGET_CLUSTER_COUNT,
)

print("architecture_name =", architecture_name)
print("experiment_name   =", experiment_name)
print("hmm_enabled       =", hmm_config.enabled)
print("target_clusters   =", TARGET_CLUSTER_COUNT)


architecture_name = encoder_only
experiment_name   = encoder_only_baseline
hmm_enabled       = True


## Batch Comparison Template

这里放整组实验设置。当前默认已经把所有 Track B 候选架构和经典聚类 baseline 都列进来了，你可以直接整批跑，也可以删减成自己这轮想比较的子集。


In [3]:
experiment_setups = build_default_experiment_setups(
    data_dir=DATA_DIR,
    hmm_enabled=HMMLEARN_AVAILABLE,
    target_cluster_count=TARGET_CLUSTER_COUNT,
)

display(pd.DataFrame({
    "name": [setup["name"] for setup in experiment_setups],
    "architecture": [setup["architecture"] for setup in experiment_setups],
    "epochs": [setup["training_config"].num_epochs for setup in experiment_setups],
}))


In [4]:
batch_results = []

for setup in experiment_setups:
    runner = ARCHITECTURE_RUNNERS[setup["architecture"]]
    result = runner(
        experiment_name=setup["name"],
        data_config=setup["data_config"],
        model_config=setup["model_config"],
        training_config=setup["training_config"],
        clustering_config=setup["clustering_config"],
        hmm_config=setup["hmm_config"],
    )
    batch_results.append(result)

comparison_df = compare_experiment_summaries(batch_results)
display(comparison_df)


[encoder_only] epoch=1/100 train_loss=0.799424 val_loss=0.660233
[encoder_only] epoch=2/100 train_loss=0.702822 val_loss=0.648311
[encoder_only] epoch=3/100 train_loss=0.674053 val_loss=0.645785
[encoder_only] epoch=4/100 train_loss=0.654702 val_loss=0.647264
[encoder_only] epoch=5/100 train_loss=0.637348 val_loss=0.640233
[encoder_only] epoch=6/100 train_loss=0.616488 val_loss=0.638403
[encoder_only] epoch=7/100 train_loss=0.617261 val_loss=0.635161
[encoder_only] epoch=8/100 train_loss=0.612349 val_loss=0.628117
[encoder_only] epoch=9/100 train_loss=0.603935 val_loss=0.631926
[encoder_only] epoch=10/100 train_loss=0.598705 val_loss=0.629094
[encoder_only] epoch=11/100 train_loss=0.579915 val_loss=0.626871
[encoder_only] epoch=12/100 train_loss=0.584932 val_loss=0.632643
[encoder_only] epoch=13/100 train_loss=0.578596 val_loss=0.629136
[encoder_only] epoch=14/100 train_loss=0.568379 val_loss=0.624073
[encoder_only] epoch=15/100 train_loss=0.569924 val_loss=0.628033
[encoder_only] epoc

,experiment_name,architecture,input_dim,window_size,n_sequence_rows,n_windows,used_sequence_features,skipped_sequence_features,best_epoch,best_val_loss,target_cluster_count,silhouette,ari_vs_hmm,nmi_vs_hmm
0,encoder_only_wider,encoder_only,8,60,2834,2775,"SPY_ret, TLT_ret, GLD_ret, UUP_ret, HYG_ret, L...",,12,0.622998,4,0.216547,0.046995,0.100479
1,encoder_only_small,encoder_only,8,60,2834,2775,"SPY_ret, TLT_ret, GLD_ret, UUP_ret, HYG_ret, L...",,19,0.615727,4,0.231047,0.037364,0.084345
2,rnn_lstm_hybrid_baseline,rnn_lstm_hybrid,8,60,2834,2775,"SPY_ret, TLT_ret, GLD_ret, UUP_ret, HYG_ret, L...",,6,0.640009,4,0.175749,0.029005,0.081020
3,pure_rnn_baseline,pure_rnn,8,60,2834,2775,"SPY_ret, TLT_ret, GLD_ret, UUP_ret, HYG_ret, L...",,9,0.637411,4,0.213243,0.002735,0.035061


## PCA Scatter of HMM and Model Classifications

这一段把 `HMM` 和当前 `batch_results` 里的所有模型分类结果都压到二维空间里画散点图：

- `HMM` 子图：对 HMM 参考特征做 PCA，按 `hmm_regime` 上色。
- 模型子图：对每个模型学到的 `embedding` 做 PCA，按 `cluster` 上色。

子图数量会随着你注册进来的架构动态扩展。这样你可以直接看出不同方法的分群形状、簇间分离程度，以及它们和 HMM 的视觉差异。


In [ ]:
hmm_source_result = next((result for result in batch_results if result.get("hmm_results") is not None), None)
if hmm_source_result is None:
    raise ValueError("No HMM reference found in batch_results. Enable HMM comparison in at least one experiment.")

common_dates = None
for result in batch_results:
    result_dates = pd.Index(result["window_end_dates"], name="Date")
    common_dates = result_dates if common_dates is None else common_dates.intersection(result_dates)

hmm_reference = hmm_source_result["hmm_results"]["hmm_reference"].copy()
common_dates = common_dates.intersection(hmm_reference.index)
common_dates = common_dates.sort_values()

if len(common_dates) == 0:
    raise ValueError("No overlapping dates found between model windows and HMM reference.")

print(f"Common aligned windows for visualization: {len(common_dates)}")


def compute_pca_projection(matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    scaled = StandardScaler().fit_transform(matrix)
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(scaled)
    return coords, pca.explained_variance_ratio_


def scatter_by_label(ax, coords: np.ndarray, labels: pd.Series, title: str, explained_ratio: np.ndarray, legend_title: str):
    label_series = labels.astype(str)
    unique_labels = sorted(label_series.unique())
    cmap = plt.cm.get_cmap("tab10", max(len(unique_labels), 1))

    for idx, label in enumerate(unique_labels):
        mask = (label_series == label).to_numpy()
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            s=12,
            alpha=0.75,
            color=cmap(idx),
            label=label,
        )

    ax.set_title(title, fontsize=11)
    ax.set_xlabel(f"PC1 ({explained_ratio[0]:.1%})")
    ax.set_ylabel(f"PC2 ({explained_ratio[1]:.1%})")
    ax.grid(alpha=0.2, linewidth=0.5)
    ax.legend(title=legend_title, fontsize=8, title_fontsize=9, loc="best")


n_panels = len(batch_results) + 1
n_cols = 3
n_rows = int(np.ceil(n_panels / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4.8 * n_rows), constrained_layout=True)
axes = np.atleast_1d(axes).flatten()

numeric_hmm_columns = [
    column
    for column in hmm_reference.columns
    if column not in {"hmm_state", "hmm_regime"} and pd.api.types.is_numeric_dtype(hmm_reference[column])
]
hmm_view = hmm_reference.loc[common_dates, numeric_hmm_columns]
hmm_coords, hmm_ratio = compute_pca_projection(hmm_view.to_numpy())
hmm_labels = hmm_reference.loc[common_dates].apply(
    lambda row: f"{int(row['hmm_state'])}: {row['hmm_regime']}",
    axis=1,
)
scatter_by_label(
    axes[0],
    hmm_coords,
    hmm_labels,
    title="HMM Reference (PCA of HMM features)",
    explained_ratio=hmm_ratio,
    legend_title="HMM regime",
)

for plot_idx, result in enumerate(batch_results, start=1):
    result_dates = pd.Index(result["window_end_dates"], name="Date")
    positions = result_dates.get_indexer(common_dates)
    if np.any(positions < 0):
        raise ValueError(f"{result['experiment_name']} is missing some common_dates alignment points.")

    embeddings = result["embeddings"][positions]
    labels = pd.Series(result["cluster_labels"][positions], index=common_dates, name="cluster")
    coords, explained_ratio = compute_pca_projection(embeddings)
    summary = result["summary"]
    title = (
        f"{summary['experiment_name']}\n"
        f"sil={summary['silhouette']:.3f}, ARI={summary['ari_vs_hmm']:.3f}, NMI={summary['nmi_vs_hmm']:.3f}"
    )
    scatter_by_label(
        axes[plot_idx],
        coords,
        labels,
        title=title,
        explained_ratio=explained_ratio,
        legend_title="Cluster",
    )

for ax in axes[n_panels:]:
    ax.axis("off")

plt.suptitle("Track B Architecture Comparison: HMM vs Model Classification Scatter", fontsize=14)
plt.show()


## PCA of Raw Input Windows Colored by Each Classification

这一段不再使用模型学到的 embedding，而是直接对 **原始输入窗口** 做 PCA：

- 先把每个 `60 x feature_dim` 窗口展平成一个向量，再压到二维。
- 然后分别按 `HMM` 和当前所有模型的分类结果上色。

这样你看到的是 **同一份原始数据空间** 里，不同分类器到底是怎么切簇的，而且子图数量会自动随着架构数量扩展。


In [ ]:
hmm_source_result = next((result for result in batch_results if result.get("hmm_results") is not None), None)
if hmm_source_result is None:
    raise ValueError("No HMM reference found in batch_results. Enable HMM comparison in at least one experiment.")

common_dates_raw = None
for result in batch_results:
    result_dates = pd.Index(result["window_end_dates"], name="Date")
    common_dates_raw = result_dates if common_dates_raw is None else common_dates_raw.intersection(result_dates)

hmm_reference_raw = hmm_source_result["hmm_results"]["hmm_reference"].copy()
common_dates_raw = common_dates_raw.intersection(hmm_reference_raw.index)
common_dates_raw = common_dates_raw.sort_values()

if len(common_dates_raw) == 0:
    raise ValueError("No overlapping dates found between model windows and HMM reference.")

raw_source_result = batch_results[0]
raw_result_dates = pd.Index(raw_source_result["window_end_dates"], name="Date")
raw_positions = raw_result_dates.get_indexer(common_dates_raw)
if np.any(raw_positions < 0):
    raise ValueError("The chosen raw_source_result is missing some common_dates alignment points.")

raw_windows = raw_source_result["windows"][raw_positions]
raw_flattened = raw_windows.reshape(len(raw_windows), -1)
raw_coords, raw_ratio = compute_pca_projection(raw_flattened)

n_panels_raw = len(batch_results) + 1
n_cols_raw = 3
n_rows_raw = int(np.ceil(n_panels_raw / n_cols_raw))
fig, axes = plt.subplots(n_rows_raw, n_cols_raw, figsize=(6 * n_cols_raw, 4.8 * n_rows_raw), constrained_layout=True)
axes = np.atleast_1d(axes).flatten()

hmm_labels_raw = hmm_reference_raw.loc[common_dates_raw].apply(
    lambda row: f"{int(row['hmm_state'])}: {row['hmm_regime']}",
    axis=1,
)
scatter_by_label(
    axes[0],
    raw_coords,
    hmm_labels_raw,
    title="HMM labels on PCA of raw input windows",
    explained_ratio=raw_ratio,
    legend_title="HMM regime",
)

for plot_idx, result in enumerate(batch_results, start=1):
    result_dates = pd.Index(result["window_end_dates"], name="Date")
    positions = result_dates.get_indexer(common_dates_raw)
    if np.any(positions < 0):
        raise ValueError(f"{result['experiment_name']} is missing some common_dates alignment points.")

    labels = pd.Series(result["cluster_labels"][positions], index=common_dates_raw, name="cluster").map(
        lambda value: f"cluster {int(value)}"
    )
    summary = result["summary"]
    title = (
        f"{summary['experiment_name']} on raw-window PCA\n"
        f"sil={summary['silhouette']:.3f}, ARI={summary['ari_vs_hmm']:.3f}, NMI={summary['nmi_vs_hmm']:.3f}"
    )
    scatter_by_label(
        axes[plot_idx],
        raw_coords,
        labels,
        title=title,
        explained_ratio=raw_ratio,
        legend_title="Cluster",
    )

for ax in axes[n_panels_raw:]:
    ax.axis("off")

plt.suptitle("Raw Input Window PCA Colored by HMM and Model Classification Results", fontsize=14)
plt.show()


## Next Step

等你后面继续实现新的架构时：

1. 新建一个和现有 runner 类似签名的实验入口。
2. 把它注册进 `build_architecture_runners()`。
3. 在 `build_default_model_config()` 和 `build_default_experiment_setups()` 里补上默认配置。

这样这个 notebook 就能继续当统一的对比面板，不用每加一个架构就重写一套实验流程。
